# 03 -- Exploratory Data Analysis

Task 5's EDA over `data/processed/documents.csv`. All the actual
computations (class counts, length stats, word frequencies) live in
`src/dataset/eda.py` -- this notebook calls them and adds the
visualizations/narrative that belong in a review artifact, not in
production code.

## Project setup and imports

In [ ]:
import sys
from pathlib import Path

# Notebooks live in notebooks/, but the reusable pipeline code lives in
# src/ at the project root. Jupyter sets the working directory to wherever
# it was launched from, which isn't reliable -- so we search upward for
# requirements.txt (a stable marker of the project root) instead of
# hardcoding "..".
def find_project_root(marker="requirements.txt"):
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError("Could not locate project root (no requirements.txt found above cwd)")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


In [ ]:
from src.dataset import eda

eda


## Loading `data/processed/documents.csv`

In [ ]:
df = eda.load()
df.head()


## Dataset shape and columns

In [ ]:
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.dtypes


## Missing values

In [ ]:
df.isna().sum()


## Class distribution

In [ ]:
counts = eda.class_counts(df)
counts


In [ ]:
counts.plot(kind="bar", title="Documents per class");


## Documents per source agency

In [ ]:
agency_counts = eda.agency_counts(df)
agency_counts


In [ ]:
agency_counts.plot(kind="bar", title="Documents per agency");


## Text-length statistics

In [ ]:
eda.text_length_stats(df)


In [ ]:
df["text"].str.split().str.len().plot(kind="hist", bins=30, title="Word count per document");


## Duplicate detection

Exact-text duplicates are usually a sign the crawler picked up the same
PDF from two different links on an agency's site (see the URL-normalization
step in `01_crawler_research.ipynb`).

In [ ]:
dupes = eda.duplicate_rows(df)
print(f"{len(dupes)} duplicate rows")
dupes[["file_path", "source_agency", "label"]].head(20)


## Simple visualizations: most common words by category

In [ ]:
top_words = eda.top_words_by_category(df)
for label, words in top_words.items():
    print(f"\n{label}:")
    print(words)


## Data-quality observations and recommendations

Fill this in after running the cells above on the real, collected corpus:

- Class balance: is any category under-represented enough to need the
  synthetic-document augmentation mentioned in the project intro?
- Text length: any extreme outliers worth inspecting by hand (e.g. a
  mis-extracted PDF, or a form that's mostly blank fields)?
- Duplicates: how many, and are they true duplicates or near-duplicates
  (e.g. a form and its Spanish-language version)?
- Anything here that should be written into `docs/dataset_summary.md`
  before handoff to the BERT/RAG teams.